# MODE — Sequential all-dataset experiments (per-barycenter split)

Runs **CODEF Experiments 1–3 (CPU/RAM) and Raspberry Pi (CPU/RAM)** one at a time.

**Outputs:** `mode_all_raw_rmse.csv` (one row per held-out test profile, all run-level metrics repeated for plotting), `mode_runs_summary.csv` (one row per completed run), `mode_runs_errors.csv` (failed/skipped datasets), and optional per-run prediction/target arrays. Results are written **after each dataset**, not just at the end. A rerun skips successfully completed datasets unless `FORCE_RERUN=True`.


In [ ]:
# 1. Imports and select the correct MODE_utilities implementation
from pathlib import Path
import os, sys, json, time, random, traceback, importlib, tempfile
import numpy as np
import pandas as pd

# Copy MODE_utilities_per_barycenter_split.py to the same directory as this
# notebook, or set CUSTOM_MODULE_PATH to its full file path.
CUSTOM_MODULE_PATH = Path("MODE_utilities.py")
if not CUSTOM_MODULE_PATH.is_file():
    raise FileNotFoundError(
        f"Required patched utilities missing: {CUSTOM_MODULE_PATH.resolve()}\n"
        "Place MODE_utilities.py alongside this notebook."
    )

import importlib.util
spec = importlib.util.spec_from_file_location("MODE_utilities_batch_per_barycenter", str(CUSTOM_MODULE_PATH.resolve()))
AS = importlib.util.module_from_spec(spec)
spec.loader.exec_module(AS)

# Guard against accidentally importing the old concatenation-first version.
import inspect
assert "val_series_length=series_length - split_idx" in inspect.getsource(AS.train_model_from_barycenters), (
    "Wrong MODE utilities version: per-barycenter splitting is not available."
)
print("Using MODE utilities:", CUSTOM_MODULE_PATH.resolve())
print("PyTorch device:", AS.device)

In [ ]:
# 2. Original MODE settings — copied from Running_MODE(6).ipynb
POPULATION = 20
OFFSPR = 10
GENS = 10
GAMMA = 1.0
NORMALIZE = True
SERIES_LENGTH = 500
MIN_CLUSTER_SIZE = 2
USE_DIVERGENCE = False
SPLIT = 0.80
MAX_ITER = 50
BARYCENTER_MAX_ITER = 50
D_STAR_TP = 2.6
ASF_WEIGHTS_FULL = np.array([0.6, 0.2, 0.2])
TRACK_ENERGY = True  # Same as original notebook; requires codecarbon installed.

# Dataset root folders; change them if your directory structure differs.
CODEF_FOLDER = Path("CODEF")
RASPBERRY_FOLDER = Path("Rasbery_Pi")  # Exact default spelling in MODE_utilities.py
RASPBERRY_MAX_SERIES = 60

# Execution controls
OUTPUT_DIR = Path("Experimental results") / "MODE_CODEF_RPi_8_per_barycenter"
FORCE_RERUN = False
SAVE_PREDICTION_ARRAYS = True  # .npz per dataset, for future time-series plotting
FAIL_FAST = False  # False: record failure and continue with remaining datasets
SEED = None  # None preserves the stochastic setup of your original notebook

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUTPUT_DIR.resolve())

## Experiment plan — eight runs

Run CODEF Experiments I–III (CPU and RAM: six runs), then Raspberry Pi CPU and RAM (two runs). The CODEF loader reproduces `read_experiment_data(experiment, metric)` and groups observations into length-500 profiles. Raspberry Pi uses `read_raspberry_pi_dataset()` with the original first-60-profile limit. The held-out test set is the final 100 observations of each 500-point profile. Missing inputs are logged, and completed runs are saved independently.


In [ ]:
# 3. Exactly eight datasets, run sequentially
DATASETS = [
    {"id": f"CODEF_Exp{exp}_{metric}", "source": "CODEF", "experiment": exp, "metric": metric}
    for exp in (1, 2, 3) for metric in ("CPU", "RAM")
] + [
    {"id": "RPi_CPU", "source": "RPi", "column": "cpu", "metric": "CPU"},
    {"id": "RPi_RAM", "source": "RPi", "column": "memory", "metric": "RAM"},
]

assert len(DATASETS) == 8
assert len(DATASETS) == len({d["id"] for d in DATASETS})
print(pd.DataFrame(DATASETS).fillna("").to_string(index=False))


In [ ]:
# 4. File helpers and dataset loading
DETAIL_CSV = OUTPUT_DIR / "mode_all_raw_rmse.csv"
SUMMARY_CSV = OUTPUT_DIR / "mode_runs_summary.csv"
ERROR_CSV = OUTPUT_DIR / "mode_runs_errors.csv"


def atomic_csv(df, path):
    """Write via a temporary file; keep earlier completed results if interrupted."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix="._mode_", suffix=".csv", dir=path.parent)
    os.close(fd)
    try:
        df.to_csv(temp_name, index=False)
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)


def read_existing(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def load_dataset(spec):
    if spec["source"] == "CODEF":
        data_flat = AS.read_experiment_data(
            spec["experiment"], spec["metric"], base_folder=str(CODEF_FOLDER)
        )
        n = len(data_flat) // SERIES_LENGTH
        if n == 0:
            raise ValueError(f"No complete CODEF profiles in {spec['id']}")
        data = np.asarray(data_flat[:n * SERIES_LENGTH], dtype=np.float64).reshape(n, SERIES_LENGTH)
    elif spec["source"] == "RPi":
        data = AS.read_raspberry_pi_dataset(
            series_length=SERIES_LENGTH,
            base_folder=str(RASPBERRY_FOLDER),
            column=spec["column"],
        )
        data = data[:RASPBERRY_MAX_SERIES]
    else:
        raise ValueError(f"Unknown source: {spec['source']}")

    data = np.asarray(data, dtype=np.float64)
    if data.ndim != 2 or data.shape[1] != SERIES_LENGTH or data.shape[0] == 0:
        raise ValueError(f"Invalid dataset shape for {spec['id']}: {data.shape}")
    if not np.isfinite(data).all():
        raise ValueError(f"Nonfinite data found in {spec['id']}")
    test_start = int(SPLIT * SERIES_LENGTH)
    offline_data, online_data = data[:, :test_start], data[:, test_start:]
    return offline_data, online_data


def replace_dataset_rows(existing, new, dataset_id):
    if existing.empty:
        return new.copy()
    if "dataset_id" not in existing.columns:
        raise ValueError("Existing CSV is missing dataset_id; inspect before overwriting")
    return pd.concat([existing.loc[existing["dataset_id"] != dataset_id], new], ignore_index=True)


In [ ]:
# 5. One complete MODE run and independent raw-scale test evaluation

def run_one_dataset(spec):
    dataset_id = spec["id"]
    offline_data, online_data = load_dataset(spec)
    n_series, development_length = offline_data.shape
    heldout_length = online_data.shape[1]
    print(f"Loaded {dataset_id}: {n_series} profiles x {SERIES_LENGTH} points; "
          f"development={development_length}, test={heldout_length}")

    if SEED is not None:
        random.seed(SEED)
        np.random.seed(SEED)
        AS.torch.manual_seed(SEED)
        if AS.torch.cuda.is_available():
            AS.torch.cuda.manual_seed_all(SEED)

    # Exactly the original ApplyingMODE + CodeCarbon wrapper.
    (mode_result, energy) = AS.run_with_energy_tracking(
        func=lambda: AS.ApplyingMODE(
            data=offline_data,
            num_population=POPULATION,
            num_offsprings=OFFSPR,
            num_generations=GENS,
            gamma=GAMMA,
            max_iter=MAX_ITER,
            barycenter_max_iter=BARYCENTER_MAX_ITER,
            normalize=NORMALIZE,
            use_divergence=USE_DIVERGENCE,
            min_cluster_size=MIN_CLUSTER_SIZE,
            split=SPLIT,
            D_star_tp=D_STAR_TP,
        ),
        label=f"MODE_{dataset_id}",
        enable=TRACK_ENERGY,
    )
    results, problem, initial_duration = mode_result
    timing = dict(problem.timing)

    best_instance, best_model, chromosome, info = AS.ApplyingASF(
        results_of_tuning=results,
        Problem_class=problem,
        weights=ASF_WEIGHTS_FULL,
        return_full=True,
    )
    k = int(chromosome["num_clusters"])
    represented_points = k * development_length
    actual_training_points = k * int(SPLIT * development_length)
    internal_validation_points = represented_points - actual_training_points

    # Evaluate each independent held-out profile in its ORIGINAL (raw) scale.
    rows, arrays = [], {}
    for series_id, ts in enumerate(online_data):
        result = AS.evaluate_one_individual_series(
            ML=best_instance,
            series_raw=ts,
            sdtw_normalize=NORMALIZE,
            return_raw_scale=True,
        )
        rows.append({
            "dataset_id": dataset_id,
            "source": spec["source"],
            "metric": spec["metric"],
            "experiment": spec.get("experiment", ""),
            "series_id": int(series_id),
            "rmse_raw": float(result["rmse_raw"]),
            "rmse_normalized": float(result["rmse_used"]),
            "nrmse_raw_percent": float(result["nrmse_percent_raw"]),
            "n_test_predictions": len(result["targets_raw"]),
        })
        if SAVE_PREDICTION_ARRAYS:
            arrays[f"s{series_id:05d}_pred"] = np.asarray(result["preds_raw"], dtype=np.float64)
            arrays[f"s{series_id:05d}_target"] = np.asarray(result["targets_raw"], dtype=np.float64)
            arrays[f"s{series_id:05d}_index"] = np.asarray(result["prediction_time_index"], dtype=int)

    df = pd.DataFrame(rows)
    if len(df) != n_series or df["rmse_raw"].isna().any():
        raise RuntimeError("Incomplete raw RMSE evaluation; no results will be committed")

    # 1 row per dataset (run-level metrics). All durations are seconds.
    summary = {
        "dataset_id": dataset_id,
        "source": spec["source"],
        "metric": spec["metric"],
        "experiment": spec.get("experiment", ""),
        "n_series": int(n_series),
        "original_series_length": int(SERIES_LENGTH),
        "development_series_length": int(development_length),
        "test_series_length": int(heldout_length),
        "selected_k": k,
        "input_representation_points": int(represented_points),
        "lstm_training_points": int(actual_training_points),
        "internal_validation_points": int(internal_validation_points),
        "full_development_points": int(n_series * development_length),
        "data_reduction_percent": float(100 * (1 - represented_points / (n_series * development_length))),
        "total_mode_sec": float(timing["total_mode_sec"]),
        "clustering_sec": float(timing["clustering_sec"]),
        "training_sec": float(timing["training_sec"]),
        "validation_sec": float(timing["validation_sec"]),
        "stability_sec": float(timing["stability_sec"]),
        "other_sec": float(timing["other_sec"]),
        "measured_components_sec": float(timing["measured_components_sec"]),
        "n_clusterings": int(timing["n_clusterings"]),
        "n_trainings": int(timing["n_trainings"]),
        "n_validations": int(timing["n_validations"]),
        "n_stability_evals": int(timing["n_stability_evals"]),
        "clustering_cache_hits": int(timing["clustering_cache_hits"]),
        "energy_tracking_duration_sec": float(energy.get("duration_sec", np.nan)),
        "energy_wh": float(energy.get("energy_wh", np.nan)),
        "emissions_kg": float(energy.get("emissions_kg", np.nan)),
        "tracker_available": bool(energy.get("tracker_available", False)),
        "mean_raw_rmse": float(df["rmse_raw"].mean()),
        "median_raw_rmse": float(df["rmse_raw"].median()),
        "std_raw_rmse_sample": float(df["rmse_raw"].std(ddof=1)),
        "variance_raw_rmse_sample": float(df["rmse_raw"].var(ddof=1)),
        "mean_normalized_rmse": float(df["rmse_normalized"].mean()),
        "hpo_population": int(POPULATION),
        "hpo_offspring": int(OFFSPR),
        "hpo_generations": int(GENS),
        "split": float(SPLIT),
        "cp_threshold": float(D_STAR_TP),
        "gamma": float(GAMMA),
        "max_cluster_iterations": int(MAX_ITER),
        "max_barycenter_iterations": int(BARYCENTER_MAX_ITER),
        "min_cluster_size": int(MIN_CLUSTER_SIZE),
        "use_divergence": bool(USE_DIVERGENCE),
        "normalized": bool(NORMALIZE),
        "validation_split_method": "chronological_per_barycenter",
    }
    for name in ("window", "learning_rate", "batch_size", "hidden_dim", "epochs", "layer_dim"):
        summary[f"selected_{name}"] = chromosome[name]
    for idx, weight in enumerate(ASF_WEIGHTS_FULL, 1):
        summary[f"asf_weight_{idx}"] = float(weight)

    for kk, duration in timing.get("clustering_by_k_sec", {}).items():
        summary[f"clustering_k{int(kk)}_sec"] = float(duration)

    # Repeated run-level metrics in the master file (plot by source/metric without joins).
    for col, value in summary.items():
        if col not in df.columns:
            df[col] = value
    return df, pd.DataFrame([summary]), arrays


In [ ]:
# 6. Sequential execution + save each run immediately + resumability
# This cell may run for many hours. Outputs of completed datasets survive interruption.
# If you interrupt the kernel, execute setup cells 1–5 again, then rerun this cell.

for run_idx, spec in enumerate(DATASETS, 1):
    dataset_id = spec["id"]
    existing_summary = read_existing(SUMMARY_CSV)
    existing_raw = read_existing(DETAIL_CSV)
    if not FORCE_RERUN and not existing_summary.empty and dataset_id in existing_summary["dataset_id"].values:
        n_expected = int(existing_summary.loc[existing_summary["dataset_id"] == dataset_id, "n_series"].iloc[-1])
        n_observed = int((existing_raw["dataset_id"] == dataset_id).sum()) if not existing_raw.empty else 0
        if n_observed == n_expected:
            print(f"[{run_idx}/{len(DATASETS)}] SKIP completed: {dataset_id} ({n_observed} raw RMSE values)", flush=True)
            continue

    print("\n" + "=" * 76)
    print(f"[{run_idx}/{len(DATASETS)}] RUN {dataset_id}", flush=True)
    started = time.perf_counter()
    try:
        detail, summary, prediction_arrays = run_one_dataset(spec)

        # Commit raw data first, then the summary used as resume marker.
        atomic_csv(replace_dataset_rows(read_existing(DETAIL_CSV), detail, dataset_id), DETAIL_CSV)
        atomic_csv(replace_dataset_rows(read_existing(SUMMARY_CSV), summary, dataset_id), SUMMARY_CSV)
        if SAVE_PREDICTION_ARRAYS:
            np.savez_compressed(OUTPUT_DIR / f"{dataset_id}_raw_predictions.npz", **prediction_arrays)

        info = summary.iloc[0]
        print(f"SAVED {dataset_id}: {len(detail)} raw RMSE values | "
              f"MODE time {info['total_mode_sec']:.2f} s | "
              f"clustering {info['clustering_sec']:.2f} s | "
              f"mean raw RMSE {info['mean_raw_rmse']:.4f}", flush=True)
    except Exception as exc:
        error_row = pd.DataFrame([{
            "dataset_id": dataset_id,
            "source": spec["source"],
            "metric": spec["metric"],
            "error_type": type(exc).__name__,
            "error": str(exc),
            "elapsed_before_error_sec": time.perf_counter() - started,
            "traceback": traceback.format_exc(),
        }])
        previous_errors = read_existing(ERROR_CSV)
        atomic_csv(pd.concat([previous_errors, error_row], ignore_index=True), ERROR_CSV)
        print(f"FAILED {dataset_id}: {type(exc).__name__}: {exc}", flush=True)
        if FAIL_FAST:
            raise

print("\nFinished processing dataset plan.")
print("Master raw RMSE:", DETAIL_CSV.resolve())
print("Run summary:    ", SUMMARY_CSV.resolve())
print("Failed/skipped:  ", ERROR_CSV.resolve() if ERROR_CSV.exists() else "none")

In [ ]:
# 7. Results preview — no retraining or HPO
summary = read_existing(SUMMARY_CSV)
raw = read_existing(DETAIL_CSV)

if summary.empty:
    print("No completed runs yet. Check mode_runs_errors.csv or input paths.")
else:
    display_cols = [
        "dataset_id", "n_series", "selected_k", "total_mode_sec",
        "clustering_sec", "training_sec", "validation_sec", "stability_sec", "other_sec",
        "input_representation_points", "lstm_training_points",
        "mean_raw_rmse", "std_raw_rmse_sample", "energy_wh",
    ]
    print(summary[display_cols].to_string(index=False))
    print(f"\nTotal completed runs: {len(summary)}")
    print(f"Total raw-scale test RMSE values: {len(raw)}")
    print(f"Written: {DETAIL_CSV.resolve()}")
